In [2]:
library(dplyr)
library(randomForest)

Warning message:
"le package 'dplyr' a été compilé avec la version R 4.2.3"

Attachement du package : 'dplyr'


Les objets suivants sont masqués depuis 'package:stats':

    filter, lag


Les objets suivants sont masqués depuis 'package:base':

    intersect, setdiff, setequal, union


randomForest 4.7-1.1

Type rfNews() to see new features/changes/bug fixes.


Attachement du package : 'randomForest'


L'objet suivant est masqué depuis 'package:dplyr':

    combine




In [3]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

We try to perform a k-fold CV with normal dataset, undersampled dataset and oversampled dataset to assess if the imbalance of target values affects the performance of RF.

In [4]:
datam<-read.csv("data_target_encoding.csv",stringsAsFactors = T)

target_variable<-match('damage_grade', colnames(datam))
nfeat <- ncol(datam)-1 #I remove 2 to remove building_id and damage_grade

nrows<-nrow(datam)

k<-10
n_trees<-20

In [4]:
set.seed(2) 
accuracy_vec <- array(0,k)


# 1. Shuffle the dataset randomly.
datam_idx <- sample(1:nrows)
# 2. Split the dataset into k groups
max <- ceiling(nrow(datam)/k)
splits <- split(datam_idx, ceiling(seq_along(datam_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){

    #3.1 Take the group as a hold out or test data set
    test_data <- datam[splits[[i]],]

    #3.2 Take the remaining groups as a training data set
    train_data <- datam[-splits[[i]],]   

    model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,keep.forest=TRUE,importance=TRUE)
    yhat<-predict(model,test_data[,-c(target_variable)])                      
    accuracy_vec[i]<-F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat)
    setTxtProgressBar(pb, i)
    rm('model')
    print(paste("F1-Score Micro -",i,"fold:",accuracy_vec[i]))
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
print(paste("Mean F1-Score Micro:",mean(accuracy_vec)))

  |=======                                                               |  10%[1] "F1-Score Micro - 1 fold: 0.754345573845977"
  |==============                                                        |  20%[1] "F1-Score Micro - 2 fold: 0.751390967345843"
  |=====================                                                 |  30%[1] "F1-Score Micro - 3 fold: 0.748474732358697"
  |============================                                          |  40%[1] "F1-Score Micro - 4 fold: 0.748513103871686"
  |===================================                                   |  50%[1] "F1-Score Micro - 5 fold: 0.751429338858831"
  |==========================================                            |  60%[1] "F1-Score Micro - 6 fold: 0.755726948313572"
  |=================================================                     |  70%[1] "F1-Score Micro - 7 fold: 0.753539772073213"
  |========================================================              |  80%[1] "F1-Score Micro - 8 f

In [6]:
set.seed(2)
accuracy_vec <- array(0,k)


# 1. Shuffle the dataset randomly and undersample
data_idx <- sample(1:nrow(datam))

# 2. Split the dataset into k groups
max <- ceiling(nrow(datam)/k)
splits <- split(data_idx, ceiling(seq_along(data_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){

    #3.1 Take the group as a hold out or test data set
    test_data <- datam[splits[[i]],]

    datasub_1idx <- which(datam[-splits[[i]],]$damage_grade == 1)
    n_sub <- length(datasub_1idx)

    datasub_2idx <- sample(which(datam[-splits[[i]],]$damage_grade == 2),n_sub)
    datasub_3idx <- sample(which(datam[-splits[[i]],]$damage_grade == 3),n_sub)
    datasub <- rbind(datam[datasub_1idx,],datam[datasub_2idx,],datam[datasub_3idx,])

    #3.2 Take the remaining groups as a training data set
    train_data <- datasub 

    model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,keep.forest=TRUE,importance=TRUE)
    yhat<-predict(model,test_data[,-c(target_variable)])                      
    accuracy_vec[i]<-F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat)
    setTxtProgressBar(pb, i)
    print(paste("F1-Score Micro -",i,"fold:",accuracy_vec[i]))
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
print(paste("Mean F1-Score Micro:",mean(accuracy_vec)))

  |=======                                                               |  10%[1] "F1-Score Micro - 1 fold: 0.792026399600936"
  |==============                                                        |  20%[1] "F1-Score Micro - 2 fold: 0.791988028087948"
  |=====================                                                 |  30%[1] "F1-Score Micro - 3 fold: 0.794136832815318"
  |============================                                          |  40%[1] "F1-Score Micro - 4 fold: 0.793484517094509"
  |===================================                                   |  50%[1] "F1-Score Micro - 5 fold: 0.794750777023138"
  |==========================================                            |  60%[1] "F1-Score Micro - 6 fold: 0.797014696289475"
  |=================================================                     |  70%[1] "F1-Score Micro - 7 fold: 0.798280956218104"
  |========================================================              |  80%[1] "F1-Score Micro - 8 f

In [8]:
set.seed(2)

accuracy_vec <- array(0,k)
# 1. Shuffle the dataset randomly and undersample
data_idx <- sample(1:nrow(datam))

# 2. Split the dataset into k groups
max <- ceiling(nrow(datam)/k)
splits <- split(datam_idx, ceiling(seq_along(datam_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){
    #3.1 Take the group as a hold out or test data set
    test_data <- datam[splits[[i]],]

    dataoverid_1 <- which(datam[-splits[[i]],]$damage_grade == 1)
    dataoverid_2 <- which(datam[-splits[[i]],]$damage_grade == 2)
    dataoverid_3 <- which(datam[-splits[[i]],]$damage_grade == 3)
    n_over1 <- 0.75*(length(dataoverid_2)-length(dataoverid_1))
    n_over3 <- 0.25*(length(dataoverid_2)-length(dataoverid_3))

    dataover_1 <- sample(dataoverid_1,n_over1,replace=TRUE)
    dataover_3 <- sample(dataoverid_3,n_over3,replace=TRUE)
    dataover <- rbind(datam[-splits[[i]],],datam[-splits[[i]],][dataover_1,],datam[-splits[[i]],][dataover_3,])
     #3.2 Take the remaining groups as a training data set
    train_data <- dataover[,]   

    model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,keep.forest=TRUE,importance=TRUE)
    yhat<-predict(model,test_data[,-c(target_variable)])                      
    accuracy_vec[i]<-F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat)
    setTxtProgressBar(pb, i)
    print(paste("F1-Score Micro -",i,"fold:",accuracy_vec[i]))
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
print(paste("Mean F1-Score Micro:",mean(accuracy_vec)))

  |=======                                                               |  10%[1] "F1-Score Micro - 1 fold: 0.73696327846207"
  |==============                                                        |  20%[1] "F1-Score Micro - 2 fold: 0.733509842293082"
  |=====================                                                 |  30%[1] "F1-Score Micro - 3 fold: 0.7378841947738"
  |============================                                          |  40%[1] "F1-Score Micro - 4 fold: 0.73354821380607"
  |===================================                                   |  50%[1] "F1-Score Micro - 5 fold: 0.73757722266989"
  |==========================================                            |  60%[1] "F1-Score Micro - 6 fold: 0.739956256475193"
  |=================================================                     |  70%[1] "F1-Score Micro - 7 fold: 0.739610912858294"
  |========================================================              |  80%[1] "F1-Score Micro - 8 fold: 

In [9]:
set.seed(2)

accuracy_vec <- array(0,k)
# 1. Shuffle the dataset randomly and undersample
data_idx <- sample(1:nrow(datam))

# 2. Split the dataset into k groups
max <- ceiling(nrow(datam)/k)
splits <- split(datam_idx, ceiling(seq_along(datam_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)
# 3. For each unique group:
for (i in 1:k){
    #3.1 Take the group as a hold out or test data set
    test_data <- datam[splits[[i]],]

    dataoverid_1 <- which(datam[-splits[[i]],]$damage_grade == 1)
    dataoverid_2 <- which(datam[-splits[[i]],]$damage_grade == 2)
    dataoverid_3 <- which(datam[-splits[[i]],]$damage_grade == 3)
    n_over1 <- (length(dataoverid_2)-length(dataoverid_1))
    n_over3 <- (length(dataoverid_2)-length(dataoverid_3))

    dataover_1 <- sample(dataoverid_1,n_over1,replace=TRUE)
    dataover_3 <- sample(dataoverid_3,n_over3,replace=TRUE)
    dataover <- rbind(datam[-splits[[i]],],datam[-splits[[i]],][dataover_1,],datam[-splits[[i]],][dataover_3,])
     #3.2 Take the remaining groups as a training data set
    train_data <- dataover[,]   

    model <- randomForest(x=train_data[,-c(target_variable)],
                        y=as.factor(train_data[,c(target_variable)]),
                        ntree=n_trees,keep.forest=TRUE,importance=TRUE)
    yhat<-predict(model,test_data[,-c(target_variable)])                      
    accuracy_vec[i]<-F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat)
    setTxtProgressBar(pb, i)
    print(paste("F1-Score Micro -",i,"fold:",accuracy_vec[i]))
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
print(paste("Mean F1-Score Micro:",mean(accuracy_vec)))

  |=======                                                               |  10%[1] "F1-Score Micro - 1 fold: 0.725566939104409"
  |==============                                                        |  20%[1] "F1-Score Micro - 2 fold: 0.727715743831779"
  |=====================                                                 |  30%[1] "F1-Score Micro - 3 fold: 0.729672690994206"
  |============================                                          |  40%[1] "F1-Score Micro - 4 fold: 0.726487855416139"
  |===================================                                   |  50%[1] "F1-Score Micro - 5 fold: 0.729902920072138"
  |==========================================                            |  60%[1] "F1-Score Micro - 6 fold: 0.734123786500902"
  |=================================================                     |  70%[1] "F1-Score Micro - 7 fold: 0.733855185909981"
  |========================================================              |  80%[1] "F1-Score Micro - 8 f